In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Extraemos los precios y predicciones
lazy_with_idx = finalize_dataset_for_xgboost(X_features, y_labels, apply_zscore=False).with_row_index("row_idx")
total_rows = lazy_with_idx.select(pl.len()).collect().item()
train_cutoff = int(total_rows * 0.8)

test_lazy = lazy_with_idx.filter(pl.col("row_idx") >= train_cutoff).drop("row_idx")
test_df = test_lazy.collect().drop_nulls()

precios = test_df.select('mid_price_coordenada').to_numpy().flatten()

horizontes_evaluar = [100]
plt.figure(figsize=(9, 5.5))
fee=0.00005
for k in horizontes_evaluar:
    # Calculamos el retorno futuro real
    retorno_futuro = np.zeros(len(precios))
    retorno_futuro[:-k] = (precios[k:] - precios[:-k]) / precios[:-k]

    # -----------------------------------------------------------------
    # ESTRATEGIA MODELO REAL
    # -----------------------------------------------------------------
    beneficios_real = np.zeros(len(precios))
    beneficios_real[y_pred_custom == 2] = retorno_futuro[y_pred_custom == 2]-fee
    beneficios_real[y_pred_custom == 0] = -retorno_futuro[y_pred_custom == 0]-fee
    cum_profit_real = np.cumsum(beneficios_real)

    # -----------------------------------------------------------------
    # ESTRATEGIA AL AZAR (BENCHMARK)
    # -----------------------------------------------------------------
    # Clonamos las predicciones reales para mantener exactamente el mismo
    # volumen de trading (cuántas veces se queda en STAT, cuántas en UP/DOWN)
    y_pred_azar = y_pred_custom.copy()

    # Extraemos solo los índices donde el modelo decidió operar (0 o 2)
    indices_operativos = np.where((y_pred_custom == 0) | (y_pred_custom == 2))[0]

    # Mezclamos aleatoriamente las etiquetas operativas originales
    etiquetas_operativas_mezcladas = y_pred_custom[indices_operativos]
    np.random.shuffle(etiquetas_operativas_mezcladas)

    # Asignamos las etiquetas mezcladas al modelo de azar
    y_pred_azar[indices_operativos] = etiquetas_operativas_mezcladas

    # Calculamos el beneficio al azar
    beneficios_azar = np.zeros(len(precios))
    beneficios_azar[y_pred_azar == 2] = retorno_futuro[y_pred_azar == 2]-fee
    beneficios_azar[y_pred_azar == 0] = -retorno_futuro[y_pred_azar == 0]-fee
    cum_profit_azar = np.cumsum(beneficios_azar)

    # -----------------------------------------------------------------
    # PINTAR AMBAS CURVAS
    # -----------------------------------------------------------------
    plt.plot(cum_profit_real, color='green', linewidth=2, label=f'Modelo Real (XGBoost k={k})')
    plt.plot(cum_profit_azar, color='red', linestyle='--', alpha=0.8, label=f'Modelo al Azar (Control Random)')

# Ajustes estéticos
plt.title('Validación de Micro-Alpha: Beneficio Acumulado vs Modelo al Azar', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Eventos en el Periodo de Test (Ticks)', fontweight='bold')
plt.ylabel('Retorno Acumulado', fontweight='bold')
plt.axhline(0, color='black', linestyle=':', alpha=0.5)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Extraer precios e identificar dónde se activaron tus señales con el umbral (ts = 0.5)
precios = test_df.select('mid_price_coordenada').to_numpy().flatten()

# Obtenemos los índices exactos donde tu modelo superó el umbral de confianza
indices_up = np.where(y_pred_custom == 2)[0]
indices_down = np.where(y_pred_custom == 0)[0]

# 2. Definimos una ventana de tiempo futuro para observar cómo decae la señal
# Evaluaremos desde el tick 0 (momento de la señal) hasta 300 ticks en el futuro
max_futuro = 1100
horizontes_futuros = np.arange(0, max_futuro)

# Matrices para almacenar los retornos de cada operación a lo largo del tiempo
retornos_operaciones = []

# Procesamos las señales UP
for idx in indices_up:
    if idx + max_futuro < len(precios):
        # Retorno paso a paso desde el precio de entrada: (P_{t+h} - P_t) / P_t
        ret_h = (precios[idx : idx + max_futuro] - precios[idx]) / precios[idx]
        retornos_operaciones.append(ret_h)

# Procesamos las señales DOWN (invertimos el signo del retorno porque ganamos si baja)
for idx in indices_down:
    if idx + max_futuro < len(precios):
        ret_h = -(precios[idx : idx + max_futuro] - precios[idx]) / precios[idx]
        retornos_operaciones.append(ret_h)

# Convertimos a array de numpy para promediar
retornos_operaciones = np.array(retornos_operaciones)

# Calculamos el retorno medio esperado para cada paso en el futuro
alfa_medio_temporal = np.mean(retornos_operaciones, axis=0)

# 3. Graficar el Alpha Decay Realizado
plt.figure(figsize=(9, 5.5))
plt.plot(horizontes_futuros, alfa_medio_temporal, color='royalblue', linewidth=2.5, label='Evolución del Micro-Alpha')


# Formato y estilo
plt.title(f'Curva de Alpha Decay Empírica (Umbral ts={ts})\nTrayectoria Promedio del Precio Post-Señal', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Tiempo Transcurrido desde la Señal ($h$ Ticks)', fontweight='bold')
plt.ylabel('Retorno Medio Obtenido', fontweight='bold')
plt.axhline(0, color='black', linestyle=':', alpha=0.5)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import pandas as pd

def plot_interactive_analysis(df, n_points=10000):
    """
    Gráfico interactivo optimizado para snapshots de 100ms.
    Muestra: Midprice, Señales HFT Netas (Buy=2 / Sell=0) y el Imbalance del Libro (OBI L1).
    """
    # 1. Preparación de datos segura y conversión controlada
    if isinstance(df, pl.LazyFrame):
        df_plot = df.head(n_points).collect().to_pandas()
    else:
        df_plot = df.head(n_points).to_pandas()

    # CONVERSIÓN CRUCIAL: Pasamos los microsegundos enteros a datetime real para que Plotly pinte el eje X legible
    # Usamos la columna local_timestamp que es la que homogeneizamos en el paso de los 100ms
    df_plot["datetime_x"] = pd.to_datetime(df_plot["timestamp"], unit="us")

    # 2. Crear Subplots
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.7, 0.3],
        subplot_titles=("Mid Price & HFT Triple Barrier Labels", "Order Book Imbalance (L1)")
    )

    # --- TRAZO DE PRECIO (Corregido a 'midprice') ---
    fig.add_trace(go.Scatter(
        x=df_plot["datetime_x"],
        y=df_plot["mid_price_coordenada"],
        mode='lines',
        name='Mid Price',
        line=dict(color='#8c8c8c', width=1.5),
        opacity=0.8
    ), row=1, col=1)

    # --- ETIQUETAS DE COMPRA (Corregido a la clase 2) ---
    buys = df_plot[df_plot["label_k_100"] == 2]
    fig.add_trace(go.Scatter(
        x=buys["datetime_x"],
        y=buys["mid_price_coordenada"],
        mode='markers',
        name='BUY Signal (Class 2)',
        marker=dict(
            symbol='triangle-up',
            size=11,
            color='#00ffcc',
            line=dict(width=1, color='black')
        ),
        hovertemplate="<b>HFT BUY (TP Breached)</b><br>Price: %{y}<br>%{x}<extra></extra>"
    ), row=1, col=1)

    # --- ETIQUETAS DE VENTA (Corregido a la clase 0) ---
    sells = df_plot[df_plot["label_k_100"] == 0]
    fig.add_trace(go.Scatter(
        x=sells["datetime_x"],
        y=sells["mid_price_coordenada"],
        mode='markers',
        name='SELL Signal (Class 0)',
        marker=dict(
            symbol='triangle-down',
            size=11,
            color='#ff3366',
            line=dict(width=1, color='black')
        ),
        hovertemplate="<b>HFT SELL (SL Breached)</b><br>Price: %{y}<br>%{x}<extra></extra>"
    ), row=1, col=1)

    # --- INDICADOR INFERIOR (OBI L1 alineado en el tiempo) ---
    fig.add_trace(go.Scatter(
        x=df_plot["datetime_x"],
        y=df_plot["OBI_Nivel_1"],
        mode='lines',
        name='OBI L1',
        line=dict(color='#1f77b4', width=1),
        fill='tozeroy',
        opacity=0.6
    ), row=2, col=1)

    # 3. Layout, Estética Oscura y Sliders
    fig.update_layout(
        height=850,
        template="plotly_dark",
        title_text="HFT L2 Microstructure & Execution Analysis (100ms Snapshots)",
        hovermode="x unified",
        showlegend=True,
        xaxis2_rangeslider_visible=True,
        xaxis2_rangeslider_thickness=0.04
    )

    # Permitir que el eje Y se auto-ajuste dinámicamente al hacer zoom en el eje X
    fig.update_yaxes(fixedrange=False, row=1, col=1)
    fig.update_yaxes(fixedrange=False, row=2, col=1)

    fig.show()

In [ ]:
# =========================================================================
# SCRIPT PARA CORRER EL GRÁFICO DE AUDITORÍA DE ETIQUETAS
# =========================================================================

# 1. Supongamos que tu DataFrame final (Lazy) se llama 'dataset_final'
# Extraemos una porción secuencial inicial y la traemos a memoria con .collect()
print("⏳ Recolectando muestra de datos para el gráfico...")
dataset_completo_graph=dataset_completo
dataset_completo_graph=pl.concat([dataset_completo_graph, df_book_10.select("timestamp")], how="horizontal")

# 3. Ahora sí, eliminas de forma segura las filas nulas sobrantes
dataset_completo_graph = dataset_completo_graph.drop_nulls()
# 2. Llamamos a la función que creamos pasándole este DataFrame de Polars
# (Asegúrate de haber ejecutado antes la celda con la definición de 'plot_strict_trading_labels')
print("🎨 Generando gráfico interactivo en Plotly...")
plot_interactive_analysis(dataset_completo_graph,20000)